# <h1><center>Introduction to Machine Learning and Data Science</center></h1>
# <h1><center>Assignment 4</center></h1>
### <h1><center>Important NOTE: In order to get full credit, for every question, you need to provide the details of your work on how to get to a solution or the end of the proof.</center></h1>
### <h1><center>Instructor: Tan Bui-Thanh</center></h1>
### <h1><center>TA: Cevahir Koprulu</center></h1>
#### <h1><center>Due: 11pm 10/30/2026</center></h1>
### Submit a single pdf. All code outputs should be reported in the pdf.

Download the latest version of the lecture notes before starting. Question and exercise numbering refers to that version.

A subset of these questions will be graded. The subset is not announced in advance, so answer all twelve.

In this class, unless otherwise stated, all vectors are column vectors, and they are written in boldface fonts.

In [1]:
# --- COE 379L standard preamble ---------------------------------
# Pin versions so this notebook still runs next semester.
!pip install -q "jax[cpu]==0.4.35" "optax==0.2.3" "ucimlrepo"

import jax, jax.numpy as jnp, numpy as np, platform
import matplotlib.pyplot as plt

# Double precision. Read Section C.3 before removing this line.
jax.config.update("jax_enable_x64", True)

print("python  :", platform.python_version())
print("jax     :", jax.__version__)
print("devices :", jax.devices())
print("x64 on  :", jnp.zeros(1).dtype == jnp.float64)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


python  : 3.13.5
jax     : 0.4.35
devices : [CpuDevice(id=0)]
x64 on  : True


---

## Question 1

*Notes, Chapter 9, Exercise 9.*

Reproduce Example 9.1.1. Fit both a softmax classifier and a two class GDA to the same well separated blobs, then evaluate each at a point far outside the data. Report the posterior from both and the marginal $\mathbb{P}(\mathbf{x})$ from the GDA, and confirm the posteriors agree while only one model registers anything unusual. Then set a density threshold that rejects 1% of the training data and report how far from the means a point must lie to be rejected.

In [32]:
# Question 1: your code here

key = jax.random.PRNGKey(0)

# Get training data from a GMM with means (0,0) and (4,0) and cov 0.6I
true_means = jnp.array([[0.0,0.0], [4.0,0.0]])
true_covmat = 0.6*jnp.eye(2)

@jax.jit
def sample(key):
    key_bern, key_norm = jax.random.split(key)
    y = jax.random.bernoulli(key_bern, 0.5).astype(jnp.int32)
    x = true_means[y] + true_covmat @ jax.random.normal(key_norm, shape=(2,))/jnp.sqrt(0.6)
    return x,y

n_data = 1000
def generate_data(key, n):
    keys = jax.random.split(key, n)
    X, Y = jax.vmap(sample)(keys)
    return X, Y

X_data, y_data = generate_data(key, n_data)

# Two-class GDA
MLE_weights = jnp.array([jnp.mean(y_data == k) for k in (0,1)])
MLE_means = jnp.array([X_data[y_data == k].mean(axis=0) for k in (0,1)])
resids = X_data - MLE_means[y_data]
MLE_cov = (resids.T @ resids) / n_data

def log_gauss(x, m):
    diff = np.atleast_2d(x) - m
    return -0.5 * np.einsum("ij,jk,ik->i", diff, jnp.linalg.inv(MLE_cov), diff) - 0.5 * (X_data.shape[1] * np.log(2 * np.pi) + jnp.linalg.slogdet(MLE_cov)[1])

def gda_log_joint(x):
    return np.stack([np.log(MLE_weights[k]) + log_gauss(x, MLE_means[k]) for k in (0, 1)], axis=1)

def gda_posterior(x):
    logjoint = gda_log_joint(x)
    return np.exp(logjoint[:, 1] - np.logaddexp(logjoint[:, 0], logjoint[:, 1]))

def gda_marginal(x):
    logjoint = gda_log_joint(x)
    return np.exp(np.logaddexp(logjoint[:, 0], logjoint[:, 1]))

# Softmax Classifier 
Phi = jnp.hstack([jnp.ones((n_data, 1)), jnp.asarray(X_data)])
Y = jax.nn.one_hot(jnp.asarray(y_data), 2)
lam = 1e-3

def loss(W):                      
    logits = Phi @ W
    return -jnp.mean(jnp.sum(Y * jax.nn.log_softmax(logits), axis=1)) + 0.5 * lam * jnp.sum(W[1:] ** 2)

grad = jax.jit(jax.grad(loss))
W = jnp.zeros((3, 2))
for _ in range(5000):
    W = W - 0.5 * grad(W)

def softmax_posterior(x):
    x = np.atleast_2d(x)
    logits = np.hstack([np.ones((len(x), 1)), x]) @ np.asarray(W)
    return np.asarray(jax.nn.softmax(logits, axis=1))[:, 1]

# Evaluation
queries = np.array([[3.6, 0.2], [2.0, 0.0], [4.2, 9.0], [25.0, 0.0]])
print(f"{'x':>12} | {'softmax P(y=2|x)':>16} | {'GDA P(y=2|x)':>13} | {'GDA P(x)':>10}")
for q in queries:
    print(f"{f'({q[0]:g}, {q[1]:g})':>12} | {softmax_posterior(q)[0]:16.3f} | {gda_posterior(q)[0]:13.3f} | {gda_marginal(q)[0]:10.2e}")
print("density at a class centre:", gda_marginal(MLE_means[1])[0])

# Density Threshold Rejection
p_train = gda_marginal(X_data)                       # P(x) at every training point
tau = np.quantile(p_train, 0.01)                     # 1st percentile -> 1% fall below it
print(f"\nthreshold tau = {tau:.3e}  (rejects {np.mean(p_train < tau):.1%} of training points)")

# How far from a mean must a point be to be rejected?
r = np.linspace(0, 6, 60001)
directions = {"+x1 (away from other class)": [1, 0], "+x2 (perpendicular)": [0, 1],
              "-x2 (perpendicular)": [0, -1], "diagonal": [1/np.sqrt(2), 1/np.sqrt(2)]}
for k in (0, 1):
    for name, u in directions.items():
        u = np.array(u, float)
        if k == 0: u[0] = -u[0]                      # for class 1, "away" is the -x1 direction
        pts = np.asarray(MLE_means[k]) + r[:, None] * u
        r_rej = r[np.argmax(gda_marginal(pts) < tau)]
        print(f"class {k+1}, along {name:>28}: rejected beyond distance {r_rej:.3f}")

           x | softmax P(y=2|x) |  GDA P(y=2|x) |   GDA P(x)
  (3.6, 0.2) |            0.999 |         1.000 |   1.20e-01
      (2, 0) |            0.481 |         0.588 |   8.98e-03
    (4.2, 9) |            1.000 |         1.000 |   4.90e-31
     (25, 0) |            1.000 |         1.000 |  2.97e-165
density at a class centre: 0.1382399003738432

threshold tau = 1.998e-03  (rejects 1.0% of training points)
class 1, along  +x1 (away from other class): rejected beyond distance 2.216
class 1, along          +x2 (perpendicular): rejected beyond distance 2.233
class 1, along          -x2 (perpendicular): rejected beyond distance 2.233
class 1, along                     diagonal: rejected beyond distance 2.218
class 2, along  +x1 (away from other class): rejected beyond distance 2.231
class 2, along          +x2 (perpendicular): rejected beyond distance 2.248
class 2, along          -x2 (perpendicular): rejected beyond distance 2.248
class 2, along                     diagonal: rejected b

---

## Question 2

*Notes, Chapter 9, Exercise 10.*

Reproduce the conditioning table of Example 9.4.1. Draw $n_k$ samples from $\mathcal{N}(\mathbf{0}, \mathrm{I})$ in $d = 10$, form the sample covariance, and record its condition number, repeating enough times for a stable median. Confirm the singular case at $n_k \le d$ and the decay thereafter. How large must $n_k/d$ be before $\kappa < 2$?

In [67]:
# Question 2: your code here

key = jax.random.PRNGKey(0)
d = 10 # dimension

# Sample from Gaussian
sample = lambda key: jax.random.normal(key, shape=(d,))

def get_condition_number(key, n):
    keys = jax.random.split(key, n)
    samples = jax.vmap(sample)(keys)
    cov = jnp.cov(samples.T)
    kappa = jnp.linalg.cond(cov)

    return kappa

def batch_kappa(key, n, reps=2000):
    key_reps = jax.random.split(key, reps)
    kappa_ests = jax.vmap(get_condition_number, in_axes=(0,None))(key_reps, n)
    return kappa_ests

for i, nk in enumerate([5, 10, 11, 15, 100, 200, 225, 238, 239, 240, 250, 500, 1000]):
    kappas = batch_kappa(jax.random.fold_in(key, i), nk)
    print(f"Median Condition Number for {nk} samples: {jnp.median(kappas)}")

Median Condition Number for 5 samples: 3.802010076264629e+17
Median Condition Number for 10 samples: 4.977435190365003e+16
Median Condition Number for 11 samples: 975.6089736975646
Median Condition Number for 15 samples: 51.03899881885505
Median Condition Number for 100 samples: 2.9992441582740685
Median Condition Number for 200 samples: 2.142445982548468
Median Condition Number for 225 samples: 2.0518292253540986
Median Condition Number for 238 samples: 2.006042841738469
Median Condition Number for 239 samples: 2.006841468697532
Median Condition Number for 240 samples: 2.002765952970353
Median Condition Number for 250 samples: 1.9791208309084327
Median Condition Number for 500 samples: 1.613563292146074
Median Condition Number for 1000 samples: 1.3973825645668088


---

## Question 3

*Notes, Chapter 9, Exercise 13.*

Show that if all classes share a covariance, $\Sigma_k = \Sigma$, the quadratic terms cancel and the boundary $\mathbb{P}(y = j \mid \mathbf{x}) = \mathbb{P}(y = k \mid \mathbf{x})$ is a hyperplane, as in Figure 9.3a. Show further that the induced $\mathbb{P}(y \mid \mathbf{x})$ has exactly the logistic form of Chapter 8.

---

## Question 4

*Notes, Chapter 10, Exercise 22.*

Reproduce Figure 10.1. Using (10.9), confirm $\kappa(\mathbf{H}) \approx 6.8$. From $\boldsymbol{\theta}^0 = (-1.6, 2.2)$ run 80 steps of full batch gradient descent and 80 of Algorithm 6 at $|\mathcal{B}| = 1$, both at $\alpha$, and plot the two paths over level sets of $J$. Report $\|\boldsymbol{\theta}^{80} - \boldsymbol{\theta}^\star\|$ for each (roughly 0.15 and 0.6) and count the sample evaluations each consumed. Then answer: is the stochastic path's failure to settle a property of the step size, of the batch size, or of both?

In [ ]:
# Question 4: your code here


---

## Question 5

*Notes, Chapter 10, Exercise 23.*

Reproduce Figure 10.2. Fix the probe point $\boldsymbol{\theta} = \boldsymbol{\theta}^\star + (0.35, -0.25)$. For $|\mathcal{B}| = 1, 2, 4, \dots, 256$ estimate $\mathbb{E}\|\mathbf{g}_{\mathcal{B}} - \nabla_{\boldsymbol{\theta}} J\|^2$ by resampling the batch at least 4000 times, and plot it against $|\mathcal{B}|$ on log–log axes. Fit the slope; you should get $-1.00$ to two figures. Overlay $\sigma^2/|\mathcal{B}|$ anchored at $|\mathcal{B}| = 1$ and confirm the measured points lie on it. How few resamplings can you use before the fitted slope becomes unreliable?

In [ ]:
# Question 5: your code here


---

## Question 6

*Notes, Chapter 10, Exercise 24.*

Reproduce Figure 10.3. Run Algorithm 6 at $|\mathcal{B}| = 1$ for 4000 iterations under three schedules (constant $\alpha$, constant $\alpha/4$, and $\alpha_j = \alpha/(1 + 0.008j)$), averaging $J(\boldsymbol{\theta}^j) - J(\boldsymbol{\theta}^\star)$ over at least 40 independent runs. Plot all three on log–log axes. Confirm the two constant schedules flatten onto floors near $4.6 \times 10^{-2}$ and $8.0 \times 10^{-3}$ while the decaying one keeps descending, reaching about $1.2 \times 10^{-3}$ at $j = 4000$. Proposition 10.5.1 suggests the floors should fall in proportion to $\alpha$, so the measured ratio ought to be 4; it is closer to 5.8. Account for the discrepancy.

*Hint:* two separate effects contribute, and you can separate them numerically. First, even for a quadratic with noise of fixed covariance the exact stationary floor is $\frac{\alpha}{2}\sum_k s_k^2/(2 - \alpha\lambda_k)$ over the eigendirections of $\mathbf{H}$, which is not quite linear in $\alpha$; simulating that case gives a ratio near 4.4. Second, the gradient noise here is not of fixed covariance at all: measure $\sigma^2(\boldsymbol{\theta})$ at $\boldsymbol{\theta}^\star$ and at a point a distance 2 away and you will find it grows by more than an order of magnitude. The larger step size settles further out, where the noise is stronger, which feeds back. Proposition 10.5.1 assumes a single bound $\sigma^2$ valid everywhere, and that assumption is what is being strained.

In [ ]:
# Question 6: your code here


---

## Question 7

*Notes, Chapter 10, Exercise 26.*

Reproduce Figure 10.4. Using (10.9), run Algorithm 6 at $|\mathcal{B}| = 1, 16$ and $256$ with constant $\alpha$, averaging over at least 30 runs, and give every batch size the *same budget of sample evaluations* rather than the same number of iterations. Plot the loss gap twice, against $j$ and against $j|\mathcal{B}|$. Confirm the floors near $4.4 \times 10^{-2}$, $2.0 \times 10^{-3}$ and $1.1 \times 10^{-4}$. Then locate the interior optimum in the second plot: at a budget of 4000 evaluations which batch size is ahead, and at 96,000? Explain the crossing.

In [ ]:
# Question 7: your code here


---

## Question 8

*Notes, Chapter 10, Exercise 27.*

Reproduce Figure 10.5. Using (10.10), implement (10.4) and record the two effective steps $\alpha/(\sqrt{v_k^j} + \epsilon)$ at every iteration. Plot both against $j$ on log–log axes. Confirm that the flat coordinate is stepped between roughly 7 and 130 times further than the steep one, and that neither curve settles. Then answer: the curvature ratio is 392, so why does the step ratio never approach it?

*Hint:* $v_k$ tracks the gradient, and the gradient in coordinate $k$ is $\lambda_k$ times the error in that coordinate, not $\lambda_k$.

In [ ]:
# Question 8: your code here


---

## Question 9

*Notes, Chapter 10, Exercise 28.*

Reproduce Figure 10.6. This one needs no simulation. Plot $(1 - \beta_1^j)/\sqrt{1 - \beta_2^j}$ against $j$ for $\beta_1 = 0.9$, $\beta_2 = 0.999$, and confirm the peak of 6.6 near $j = 12$. Show analytically why the curve rises before it falls, and find where the peak moves if $\beta_2$ is lowered to 0.99. Then run Adam on (10.10) with and without the correction and report the loss gap at $j = 50$ for each. You may be surprised which is ahead, and §10.7.2 explains why that is not an argument for dropping it.

In [ ]:
# Question 9: your code here


---

## Question 10

*Notes, Chapter 10, Exercise 29.*

Reproduce Figure 10.7. Using (10.10), run SGD, momentum, RMSProp and Adam for 1500 iterations, averaged over 12 runs. Tune $\alpha$ separately for each by sweeping at least 30 values over four decades and keeping the best final loss. Do *not* share one $\alpha$, and explain in one sentence why sharing would invalidate the comparison. You should find tuned values near 0.76, 0.13, 0.015 and 0.030. Reproduce the four curves and confirm the ordering at $j = 1500$. Finally, rerun the whole comparison on the well conditioned (10.9) and report how the ranking changes.

In [ ]:
# Question 10: your code here


---

## Question 11

*Notes, Chapter 11, Exercise 15.*

Figure 11.5 shows a smooth target achieving slope $-2$ where the theorem guarantees only $-1$. Prove the sharper bound $|v - g| \le \frac{1}{8}h^2 \max|v''|$ for $v \in C^2$, and explain why Theorem 11.4.1 does not contradict it.

---

## Question 12

*Notes, Chapter 11, Exercise 18.*

Reproduce Figure 11.5. For a smooth target and for $v(x) = |x - c|$ with $c$ drawn at random, build the interpolant for $N = 2, 4, \dots, 256$, measure the maximum error, and fit the slope on log–log axes. Average the kink case over 60 values of $c$ and explain why a single $c$ gives an erratic curve.

In [ ]:
# Question 12: your code here
